In [ ]:
%sql
-- SeedFunders: the one-time seed of the Funders registry from an explicit Delta seed (the v41 table at a pinned version).
-- Run by hand only, on dev ff1013_* prefixes only; no job runs it. The production registry (openalex.funders.) was created by
-- the switch packet (notebooks/funders/release/integrated), never by this notebook. The daily loop is CreateOrUpdateFunders.
-- Idempotent: every table is created only if absent and filled only while empty, so a second run, or a run against an
-- already-seeded registry, changes nothing. Bodies come from 48d11604 CreateOrUpdateFunders cells 3-13 (ranking built once).
SELECT assert_true(trim(:prefix) RLIKE '^openalex_dev[.]rohan_lab[.]ff1013_[A-Za-z0-9_]*_$' AND trim(:allow_prod)='','dev ff1013_ prefixes only: the prod registry comes from the switch packet'),
 assert_true(trim(:run_id) RLIKE '^[A-Za-z0-9_]+$','run_id required'),
 assert_true(trim(:seed_source) RLIKE '^openalex_dev[.]rohan_lab[.]ff1013_[A-Za-z0-9_]+$'
    OR trim(:seed_source)='openalex.funders.funders_v41_table','explicit Delta seed source required'),
 assert_true(try_cast(:seed_version AS BIGINT)>=0,'explicit seed version required');


In [ ]:
%sql
-- Pin the explicit Delta seed; the served name may be a VIEW.
BEGIN
DECLARE pinned_version BIGINT;
DECLARE statement_text STRING;
SET pinned_version = CAST(:seed_version AS BIGINT);
-- VERSION AS OF rejects a VIEW and fails before registry writes if this pin is unavailable.
SET statement_text = 'CREATE OR REPLACE TEMP VIEW registry_seed_funders AS SELECT * FROM ' || trim(:seed_source) || ' VERSION AS OF ' || pinned_version;
EXECUTE IMMEDIATE statement_text;
SELECT ('fd' || trim(:run_id)) run_id, pinned_version;
END;


In [ ]:
%sql
-- create_funder
CREATE TABLE IF NOT EXISTS IDENTIFIER(trim(:prefix) || 'funder') (
  funder_id BIGINT NOT NULL, status STRING NOT NULL, merged_into_id BIGINT, ror_id STRING, ror_primary BOOLEAN, parent_id BIGINT,
  display_name STRING, alt_names ARRAY<STRING>, country_code STRING, homepage_url STRING, wikidata_id STRING, description STRING,
  image_url STRING, image_thumbnail_url STRING, created_date TIMESTAMP, updated_date TIMESTAMP, created_by STRING, run_id STRING,
  mint_key STRING) USING DELTA;


In [ ]:
%sql
-- create_funder_id
CREATE TABLE IF NOT EXISTS IDENTIFIER(trim(:prefix) || 'funder_id') (
  id_type STRING NOT NULL, id_value STRING NOT NULL, funder_id BIGINT NOT NULL, origin_funder_id BIGINT, is_primary BOOLEAN,
  source STRING, added_at TIMESTAMP, run_id STRING) USING DELTA;


In [ ]:
%sql
-- create_source_record
CREATE TABLE IF NOT EXISTS IDENTIFIER(trim(:prefix) || 'funder_source_record') (
  source STRING NOT NULL, source_key STRING NOT NULL, name STRING, alt_names ARRAY<STRING>, country STRING, ror_id STRING,
  fundref_ids ARRAY<STRING>, replaced_by ARRAY<STRING>, record_status STRING, is_funder BOOLEAN, homepage STRING, record_json STRING,
  fingerprint STRING, first_seen_at TIMESTAMP, fetched_at TIMESTAMP, changed_run_id STRING,
  outcome STRING, reason STRING, funder_id BIGINT, decided_run_id STRING, decided_fingerprint STRING) USING DELTA;


In [ ]:
%sql
-- create_decision
CREATE TABLE IF NOT EXISTS IDENTIFIER(trim(:prefix) || 'funder_decision') (
  run_id STRING NOT NULL, decided_at TIMESTAMP, source STRING, source_key STRING, action STRING NOT NULL, funder_id BIGINT,
  ror_rid STRING, fundref STRING, display_name STRING, country STRING, evidence STRING) USING DELTA;


In [ ]:
%sql
-- seed_state: the registry is either unseeded (all four tables empty) or seeded (all four filled); a partial
-- registry stops here for a person (restore or drop it first), so a fill below never mixes old and new rows.
SELECT assert_true(((SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder_decision')) = 0) = ((SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder')) = 0)
 AND ((SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder')) = 0) = ((SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder_id')) = 0)
 AND ((SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder_id')) = 0) = ((SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder_source_record')) = 0),
 'partially seeded registry: stop for a person; nothing written') seed_state_ok,
 (SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder')) = 0 will_seed;


In [ ]:
%sql
-- seed_ranking: each seed funder's merge root (seed_root, up to 3 hops) and the funders that share one ROR id with
-- their tier, keeper and works (seed_shared_ror). Built once here; the four fills below read these two views.
CREATE OR REPLACE TEMP VIEW seed_root AS
WITH v AS (SELECT * FROM registry_seed_funders),
r AS (
  SELECT a.funder_id,
    CASE WHEN a.merge_into_id IS NULL THEN NULL WHEN b.merge_into_id IS NULL THEN a.merge_into_id
         WHEN c.merge_into_id IS NULL THEN b.merge_into_id WHEN d.merge_into_id IS NULL THEN c.merge_into_id END root
  FROM v a LEFT JOIN v b ON b.funder_id = a.merge_into_id LEFT JOIN v c ON c.funder_id = b.merge_into_id
  LEFT JOIN v d ON d.funder_id = c.merge_into_id
)
SELECT funder_id, root FROM r;
CREATE OR REPLACE TEMP VIEW seed_shared_ror AS
WITH v AS (SELECT * FROM registry_seed_funders),
rorx AS (
  SELECT lower(regexp_extract(id, '([0-9a-z]{9})/*$', 1)) rid,
    array_distinct(filter(transform(flatten(transform(filter(coalesce(external_ids, array()), e -> lower(e.type) = 'fundref'),
      e -> coalesce(e.all, array()))), x -> regexp_extract(trim(x), '^(?:(?:https?://)?(?:dx\\.)?(?:doi\\.org/)?10\\.13039/)?([0-9]{5,})$', 1)), x -> x <> '')) fundrefs,
    array_distinct(filter(transform(coalesce(transform(filter(coalesce(names, array()), n -> NOT array_contains(n.types, 'acronym')), n -> n.value), array()), x -> regexp_replace(trim(regexp_replace(lower(x), '[^\\p{L}\\p{N}]+', ' ')), '^the ', '')), x -> length(x) >= 4)) nk
  FROM openalex.institutions.ror
),
live AS (
  SELECT v.funder_id, CAST(v.crossref_id AS STRING) fr, lower(regexp_extract(v.ror_id, '([0-9a-z]{9})/*$', 1)) rid, regexp_replace(trim(regexp_replace(lower(v.display_name), '[^\\p{L}\\p{N}]+', ' ')), '^the ', '') dn, coalesce(w.works_count, 0) works,
    array_distinct(filter(transform(coalesce(concat(array(v.display_name), coalesce(from_json(v.alternate_titles, 'array<string>'), array())), array()), x -> regexp_replace(trim(regexp_replace(lower(x), '[^\\p{L}\\p{N}]+', ' ')), '^the ', '')), x -> length(x) >= 4)) nk
  FROM v LEFT JOIN (SELECT id, works_count FROM openalex.funders.funders_api) w ON w.id = v.funder_id
  WHERE v.merge_into_id IS NULL AND nullif(v.ror_id, '') IS NOT NULL
),
shared AS (SELECT rid FROM live GROUP BY rid HAVING count(*) > 1),
mem AS (
  SELECT l.*, coalesce(array_contains(x.fundrefs, l.fr), false) in_xw, coalesce(array_contains(x.nk, l.dn), false) exact,
    coalesce(arrays_overlap(l.nk, x.nk), false) name_ok
  FROM live l JOIN shared s ON s.rid = l.rid LEFT JOIN rorx x ON x.rid = l.rid
),
tier AS (SELECT *, CASE WHEN in_xw AND exact THEN 1 WHEN in_xw AND name_ok THEN 2 WHEN in_xw THEN 3 WHEN exact THEN 4
                        WHEN name_ok THEN 5 ELSE 9 END t FROM mem),
best AS (SELECT rid, min(t) bt FROM tier GROUP BY rid),
keep AS (SELECT t.rid, max_by(t.funder_id, t.works * 10000000000 - t.funder_id) keeper, count(*) n_at, max(b.bt) bt
         FROM tier t JOIN best b ON b.rid = t.rid AND t.t = b.bt WHERE b.bt < 9 GROUP BY t.rid),
shared_dec AS (
  SELECT m.funder_id, m.rid, m.t, k.keeper, coalesce(k.bt, 9) bt,
    CASE WHEN (k.n_at = 1 OR k.bt IN (1, 4)) AND m.funder_id = k.keeper THEN 'PRIMARY_SHARED_ROR'
         WHEN k.n_at = 1 OR k.bt IN (1, 4) THEN 'SECONDARY_SHARED_ROR' ELSE 'NO_ATTACH_SHARED_ROR_TIE' END d,
    coalesce((k.n_at = 1 OR k.bt IN (1, 4)) AND m.funder_id <> k.keeper AND m.exact, false) merge_candidate
  FROM tier m LEFT JOIN keep k ON k.rid = m.rid
)
SELECT s.*, l.works FROM shared_dec s JOIN live l ON l.funder_id = s.funder_id;


In [ ]:
%sql
-- seed_decisions
BEGIN
IF (SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder_decision')) = 0 THEN   -- fill only while empty (idempotent)
INSERT INTO IDENTIFIER(trim(:prefix) || 'funder_decision')
WITH v AS (SELECT * FROM registry_seed_funders),
shared_dec AS (SELECT * FROM seed_shared_ror)
SELECT ('fd' || trim(:run_id)), current_timestamp(), 'openalex_v41', CAST(s.funder_id AS STRING), s.d, s.funder_id, s.rid, v.crossref_id,
  v.display_name, v.country_code,
  to_json(named_struct('tier', s.t, 'best_tier', s.bt, 'keeper', s.keeper, 'merge_candidate', s.merge_candidate, 'works', s.works))
FROM shared_dec s JOIN v ON v.funder_id = s.funder_id;
END IF;
END;


In [ ]:
%sql
-- seed_funder
BEGIN
IF (SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder')) = 0 THEN   -- fill only while empty (idempotent)
INSERT INTO IDENTIFIER(trim(:prefix) || 'funder')
WITH v AS (SELECT * FROM registry_seed_funders),
r AS (SELECT funder_id, root FROM seed_root),
shared_dec AS (SELECT * FROM seed_shared_ror)
SELECT v.funder_id, CASE WHEN v.merge_into_id IS NULL THEN 'active' ELSE 'merged' END, r.root,
  nullif(v.ror_id, ''),   -- every funder keeps the ROR it has; only the primary one resolves the ROR id
  CASE WHEN nullif(v.ror_id, '') IS NULL OR v.merge_into_id IS NOT NULL THEN NULL WHEN s.d IN ('SECONDARY_SHARED_ROR', 'NO_ATTACH_SHARED_ROR_TIE') THEN false ELSE true END,
  CAST(NULL AS BIGINT),
  v.display_name, from_json(v.alternate_titles, 'array<string>'), v.country_code, v.homepage_url, v.wikidata_id, v.description,
  v.image_url, v.image_thumbnail_url, coalesce(try_cast(substr(v.created_date, 1, 19) AS TIMESTAMP), v.updated_date), v.updated_date,
  'seed_v41', ('fd' || trim(:run_id)), CAST(NULL AS STRING)
FROM v JOIN r ON r.funder_id = v.funder_id LEFT JOIN shared_dec s ON s.funder_id = v.funder_id
UNION ALL
SELECT funder_id, 'withdrawn', CAST(NULL AS BIGINT), nullif(ror_id, ''), CAST(NULL AS BOOLEAN), CAST(NULL AS BIGINT), get_json_object(before_image_json, '$.display_name'),
  from_json(get_json_object(before_image_json, '$.alternate_titles'), 'array<string>'), get_json_object(before_image_json, '$.country_code'), get_json_object(before_image_json, '$.homepage_url'), get_json_object(before_image_json, '$.wikidata_id'),
  get_json_object(before_image_json, '$.description'), get_json_object(before_image_json, '$.image_url'), get_json_object(before_image_json, '$.image_thumbnail_url'), try_cast(substr(get_json_object(before_image_json, '$.created_date'), 1, 19) AS TIMESTAMP),
  deleted_at, 'seed_deleted', ('fd' || trim(:run_id)), CAST(NULL AS STRING)
FROM openalex.funders.deleted_funders WHERE funder_id NOT IN (SELECT funder_id FROM registry_seed_funders);
END IF;
END;


In [ ]:
%sql
-- seed_funder_id
BEGIN
IF (SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder_id')) = 0 THEN   -- fill only while empty (idempotent)
INSERT INTO IDENTIFIER(trim(:prefix) || 'funder_id')
WITH v AS (SELECT * FROM registry_seed_funders),
r AS (SELECT funder_id, root FROM seed_root)
SELECT 'fundref', CAST(v.crossref_id AS STRING), coalesce(r.root, v.funder_id), v.funder_id, true, 'seed_v41', current_timestamp(), ('fd' || trim(:run_id))
FROM v JOIN r ON r.funder_id = v.funder_id WHERE v.crossref_id IS NOT NULL
UNION ALL
SELECT 'fundref_tombstone', CAST(d.crossref_id AS STRING), d.funder_id, d.funder_id, true, 'seed_deleted', current_timestamp(), ('fd' || trim(:run_id))
FROM openalex.funders.deleted_funders d LEFT ANTI JOIN v ON CAST(v.crossref_id AS STRING) = CAST(d.crossref_id AS STRING)
WHERE d.crossref_id IS NOT NULL AND d.funder_id NOT IN (SELECT funder_id FROM v)
UNION ALL
SELECT 'ror', lower(regexp_extract(f.ror_id, '([0-9a-z]{9})/*$', 1)), f.funder_id, f.funder_id, true, 'seed_v41', current_timestamp(), ('fd' || trim(:run_id))
FROM IDENTIFIER(trim(:prefix) || 'funder') f WHERE f.status = 'active' AND f.ror_primary
UNION ALL   -- a merge loser's own ROR id points at its winner, unless an active funder already holds it
SELECT 'ror', x.rid, max(x.root), max(x.funder_id), false, 'seed_v41_merged', current_timestamp(), ('fd' || trim(:run_id))
FROM (SELECT lower(regexp_extract(v.ror_id, '([0-9a-z]{9})/*$', 1)) rid, r.root, v.funder_id FROM v JOIN r ON r.funder_id = v.funder_id
      WHERE v.merge_into_id IS NOT NULL AND nullif(v.ror_id, '') IS NOT NULL) x
LEFT ANTI JOIN (SELECT lower(regexp_extract(ror_id, '([0-9a-z]{9})/*$', 1)) rid FROM IDENTIFIER(trim(:prefix) || 'funder') WHERE status = 'active' AND ror_id IS NOT NULL) a ON a.rid = x.rid   -- held by an active funder, primary or not
GROUP BY x.rid HAVING count(DISTINCT x.root) = 1;
END IF;
END;


In [ ]:
%sql
-- seed_source_v41
BEGIN
IF (SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder_source_record')) = 0 THEN   -- fill only while empty (idempotent)
INSERT INTO IDENTIFIER(trim(:prefix) || 'funder_source_record')
WITH v AS (SELECT * FROM registry_seed_funders),
r AS (SELECT funder_id, root FROM seed_root),
shared_dec AS (SELECT * FROM seed_shared_ror)
SELECT 'openalex_v41', CAST(v.funder_id AS STRING), v.display_name, from_json(v.alternate_titles, 'array<string>'), v.country_code,
  nullif(v.ror_id, ''), CASE WHEN v.crossref_id IS NOT NULL THEN array(CAST(v.crossref_id AS STRING)) END,
  CASE WHEN v.merge_into_id IS NOT NULL THEN array(CAST(v.merge_into_id AS STRING)) END,
  CASE WHEN v.merge_into_id IS NULL THEN 'active' ELSE 'merged' END, true, v.homepage_url,
  to_json(struct(v.*), map('timestampFormat', "yyyy-MM-dd'T'HH:mm:ss.SSSSSSXXX")),   -- microseconds, so the compat passthrough reproduces v41 exactly
  sha2(to_json(struct(v.*)), 256), current_timestamp(), current_timestamp(), ('fd' || trim(:run_id)),
  CASE WHEN s.d = 'NO_ATTACH_SHARED_ROR_TIE' THEN 'not_attached' ELSE 'accepted' END,
  CASE WHEN s.d IS NOT NULL THEN s.d || CASE WHEN s.merge_candidate THEN ' (MERGE_CANDIDATE into F' || s.keeper || ')' ELSE '' END END,
  coalesce(r.root, v.funder_id), ('fd' || trim(:run_id)), CAST(NULL AS STRING)
FROM v JOIN r ON r.funder_id = v.funder_id LEFT JOIN shared_dec s ON s.funder_id = v.funder_id;
END IF;
END;


In [ ]:
%sql
-- create_ever_issued
CREATE TABLE IF NOT EXISTS IDENTIFIER(trim(:prefix) || 'f_ids_ever_issued') USING DELTA AS
SELECT funder_id, 'live_v41' source, try_cast(substr(created_date, 1, 19) AS TIMESTAMP) issued_at FROM registry_seed_funders
UNION ALL
SELECT funder_id, 'deleted', deleted_at FROM openalex.funders.deleted_funders WHERE funder_id NOT IN (SELECT funder_id FROM registry_seed_funders);


In [ ]:
%sql
-- successor_baseline
CREATE TABLE IF NOT EXISTS IDENTIFIER(trim(:prefix) || 'ror_successor_baseline') AS
SELECT DISTINCT lower(regexp_extract(id, '([0-9a-z]{9})/*$', 1)) predecessor_rid, lower(regexp_extract(rel.id, '([0-9a-z]{9})/*$', 1)) successor_rid, current_timestamp() seen_at
FROM openalex.institutions.ror LATERAL VIEW explode(relationships) x AS rel WHERE rel.type='successor';


In [ ]:
%sql
-- bridge_seed
-- One-time snapshot from fr1006_funder_source_record, 2026-10-09; 66 unique bridges.
CREATE TABLE IF NOT EXISTS IDENTIFIER(trim(:prefix) || 'fundref_predecessor_bridge') AS
SELECT predecessor_fundref_id, successor_rid FROM VALUES
('100001637','03nkpwc05'),
('100003931','00za53h95'),
('100012936','05w5gh811'),
('100013053','03k51z204'),
('100013426','03czfpz43'),
('100014232','021954z67'),
('100014732','01359dk79'),
('100016009','029xgej46'),
('100016012','0064m4q05'),
('100016079','02vtvez67'),
('100016488','01ahb7328'),
('100016535','0173y3036'),
('100016879','01n029866'),
('100019224','00dn43547'),
('100019796','05kb51347'),
('100021177','050103r16'),
('100021257','020stct92'),
('100021401','05nw4bs65'),
('100021487','0153tk833'),
('100021543','0447fe631'),
('100021768','04e4jbv10'),
('100021809','02rkzhe22'),
('100021996','015b0dz43'),
('100022179','0160cpw27'),
('100022187','031rnv444'),
('100022355','026ryr342'),
('100022379','03m2x1q45'),
('100022381','02tsp7733'),
('100022395','03rmrcq20'),
('100022403','045fdzr53'),
('100022464','00ty2a548'),
('100022681','05xte6x19'),
('100023153','0534jrf69'),
('100023295','00hzpzv83'),
('100029691','04x17ze61'),
('100030732','01kaqxm37'),
('501100003949','014wrcz97'),
('501100007500','04fkd7w53'),
('501100008810','03tb4gf50'),
('501100009077','04cfmv025'),
('501100009110','04se5nn61'),
('501100009296','036ccg086'),
('501100009539','007tjaq50'),
('501100009731','00xs3qr03'),
('501100011134','059ymf107'),
('501100011194','05etnz140'),
('501100011359','030d08e08'),
('501100011809','0327f3359'),
('501100011844','031tatj48'),
('501100012294','04m0ms912'),
('501100012334','023rhb549'),
('501100013082','05gfwht30'),
('501100013305','02qrm2a52'),
('501100014172','020fr6y52'),
('501100014570','03cq4gr50'),
('501100014937','00jrmpf38'),
('501100015047','026sv7t11'),
('501100015262','00dfd1509'),
('501100015277','02pnc0262'),
('501100015310','01h0zpd94'),
('501100016073','01qwh7a74'),
('501100016211','01dvabv26'),
('501100020448','04qr5t414'),
('501100022104','02grkyz14'),
('501100022938','03g60bf21'),
('501100023690','032e6b942')
AS b(predecessor_fundref_id,successor_rid);


In [ ]:
%sql
-- checks (as the daily checks, without c_never_deleted: there is no pre-run registry before a seed)
SELECT
  assert_true((SELECT count(*) FROM (SELECT ror_id FROM IDENTIFIER(trim(:prefix) || 'funder') WHERE status = 'active' AND ror_primary GROUP BY 1 HAVING count(*) > 1)) = 0,
    'ROR id primary on two active funders') c_ror_unique,
  assert_true((SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder') m LEFT JOIN IDENTIFIER(trim(:prefix) || 'funder') w ON w.funder_id = m.merged_into_id
    WHERE m.status = 'merged' AND (w.funder_id IS NULL OR w.status <> 'active')) = 0, 'merged funder not pointing at an active funder') c_merge_target,
  assert_true((SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder_id') i LEFT ANTI JOIN IDENTIFIER(trim(:prefix) || 'funder') f ON f.funder_id = i.funder_id) = 0, 'funder_id points at a missing funder') c_id_target,
  assert_true((SELECT count(*) FROM (SELECT id_type, id_value FROM IDENTIFIER(trim(:prefix) || 'funder_id') GROUP BY 1, 2 HAVING count(*) > 1)) = 0, 'identifier mapped twice') c_id_unique,
  assert_true((SELECT count(*) FROM (SELECT funder_id FROM IDENTIFIER(trim(:prefix) || 'funder') GROUP BY 1 HAVING count(*) > 1)) = 0, 'F id issued twice') c_f_unique,
  assert_true((SELECT count(*) FROM IDENTIFIER(trim(:prefix) || 'funder') f LEFT ANTI JOIN IDENTIFIER(trim(:prefix) || 'funder_id') i ON i.id_type = 'ror' AND i.id_value = lower(regexp_extract(f.ror_id, '([0-9a-z]{9})/*$', 1)) AND i.funder_id = f.funder_id
    WHERE f.status = 'active' AND f.ror_primary) = 0, 'primary ror_id without its funder_id row') c_ror_indexed;


In [ ]:
%sql
-- no_nonseed_fundref_keys
SELECT assert_true(count(*)=0,'non-seed funder has a FundRef key') c_no_mint_fundref FROM IDENTIFIER(trim(:prefix) || 'funder_id') i LEFT ANTI JOIN IDENTIFIER(trim(:prefix) || 'funder_source_record') s ON s.source='openalex_v41' AND s.source_key=CAST(i.funder_id AS STRING) WHERE i.id_type='fundref';


In [ ]:
%sql
-- publish_compat (first catalogue, only if absent; the daily loop republishes it every run). SELECT identical to the daily's.
CREATE TABLE IF NOT EXISTS IDENTIFIER(trim(:prefix) || 'funders_compat') AS
SELECT f.funder_id, CAST(get_json_object(v41.j, '$.crossref_id') AS BIGINT) crossref_id,
  get_json_object(v41.j, '$.location') location, f.display_name,
  CASE WHEN CASE WHEN v41.j IS NULL AND r.source_key IS NOT NULL THEN array_sort(array_distinct(concat(coalesce(f.alt_names, array()), coalesce(r.alt_names, array())))) ELSE f.alt_names END <=> from_json(get_json_object(v41.j, '$.alternate_titles'), 'array<string>') THEN get_json_object(v41.j, '$.alternate_titles') ELSE to_json(coalesce(CASE WHEN v41.j IS NULL AND r.source_key IS NOT NULL THEN array_sort(array_distinct(concat(coalesce(f.alt_names, array()), coalesce(r.alt_names, array())))) ELSE f.alt_names END, array())) END alternate_titles,
  get_json_object(v41.j, '$.uri') uri,
  coalesce(get_json_object(v41.j, '$.replaces'), CASE WHEN v41.j IS NULL THEN '[]' END) replaces,
  coalesce(get_json_object(v41.j, '$.replaced_by'), CASE WHEN v41.j IS NULL THEN '[]' END) replaced_by,
  get_json_object(v41.j, '$.tokens') tokens, CASE WHEN v41.j IS NOT NULL THEN get_json_object(v41.j, '$.ror_id') ELSE f.ror_id END ror_id,
  CASE WHEN f.status = 'merged' THEN f.merged_into_id END merge_into_id,
  CASE WHEN v41.j IS NOT NULL THEN CAST(get_json_object(v41.j, '$.merge_into_date') AS TIMESTAMP) WHEN f.status='merged' THEN f.updated_date END merge_into_date,
  coalesce(get_json_object(v41.j, '$.created_date'), CAST(f.created_date AS STRING)) created_date, f.wikidata_id,
  get_json_object(v41.j, '$.doi') doi,
  f.country_code, f.homepage_url, f.description, f.image_url, f.image_thumbnail_url, get_json_object(v41.j, '$.json_entity_hash') json_entity_hash, CASE WHEN v41.j IS NOT NULL THEN CAST(get_json_object(v41.j, '$.updated_date') AS TIMESTAMP) ELSE f.updated_date END updated_date
FROM IDENTIFIER(trim(:prefix) || 'funder') f
LEFT JOIN (SELECT CAST(source_key AS BIGINT) funder_id, record_json j FROM IDENTIFIER(trim(:prefix) || 'funder_source_record') WHERE source = 'openalex_v41') v41 ON v41.funder_id = f.funder_id
LEFT JOIN IDENTIFIER(trim(:prefix) || 'funder_source_record') r ON r.source = 'ror' AND r.source_key = lower(regexp_extract(f.ror_id, '([0-9a-z]{9})/*$', 1)) AND r.record_status <> 'gone'
WHERE f.status IN ('active', 'merged');
